In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.diagnostic import het_breuschpagan

In [5]:
url = 'https://github.com/git-iush/large_family/raw/refs/heads/main/data2.parquet'
df = pd.read_parquet(url)

In [6]:
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,BLACK,AIAN,API,OTHER,MULTR,HISP,EXPANSION,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,1,0,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,1,0,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [17]:
#Medicaid and HINSCAID equal and age
df['MEDICAID'] = df['HINSCAID'] - 1

df['AGE2'] = df['AGE']**2

In [18]:
#race control
race = 'BLACK + AIAN + API + HISP + OTHER'

In [19]:
#dummy variable locater
state_cols = list(df.loc[:, 'CONNETICUT':'DC'].columns)

df['STATE'] = df[state_cols].idxmax(axis = 1)

df.loc[df[state_cols].sum(axis = 1) == 0, 'STATE'] = 'FLORIDA'

In [20]:
#Interaction
f2 = f'NCHILD ~ MEDICAID * EXPANSION + WELFARE + AGE + AGE2 + {race}'

e1 = smf.ols(f2, data=df).fit()
print(e1.summary())

                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.229
Model:                            OLS   Adj. R-squared:                  0.229
Method:                 Least Squares   F-statistic:                 1.943e+04
Date:                Sun, 15 Nov 2026   Prob (F-statistic):               0.00
Time:                        13:37:05   Log-Likelihood:            -1.0601e+06
No. Observations:              720668   AIC:                         2.120e+06
Df Residuals:                  720656   BIC:                         2.120e+06
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.7351      0

In [21]:
#Medicaid effect
b = e1.params

print('Non expansion states', b['MEDICAID'])

print('Expansion states:', b['MEDICAID'] + b['MEDICAID:EXPANSION'])

Non expansion states 0.28563735190793904
Expansion states: 0.2120629665492313


In [14]:
#TTest for non expansion
print(e1.t_test('MEDICAID:EXPANSION = 0'))

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0            -0.0736      0.008     -9.267      0.000      -0.089      -0.058


In [22]:
#Hetero
bp = het_breuschpagan(e1.resid, e1.model.exog)

print('Breusch-Pagan p-value:', bp[1])

Breusch-Pagan p-value: 0.0


In [23]:
#Robust and clusterEd SE
e2 = smf.ols(f2, data=df).fit(cov_type='HC1')

e3 = smf.ols(f2, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['STATE']})

In [26]:
table2 = summary_col([e1, e2, e3],
                     model_names = ['Default SE', 'Robust (HC1)', 'Clustered (state)'],
                     
                     regressor_order = ['MEDICAID', 'EXPANSION', 'MEDICAID:EXPANSION', 'WELFARE', 'AGE', 'AGE2'],
                     
                     info_dict = {'N': lambda x: f'{int(x.nobs):,}'})
print(table2)


                   Default SE Robust (HC1) Clustered (state)
------------------------------------------------------------
MEDICAID           0.2856     0.2856       0.2856           
                   (0.0071)   (0.0078)     (0.0316)         
EXPANSION          -0.0394    -0.0394      -0.0394          
                   (0.0031)   (0.0030)     (0.0410)         
MEDICAID:EXPANSION -0.0736    -0.0736      -0.0736          
                   (0.0079)   (0.0086)     (0.0336)         
WELFARE            0.0541     0.0541       0.0541           
                   (0.0098)   (0.0125)     (0.0382)         
AGE                0.1824     0.1824       0.1824           
                   (0.0008)   (0.0008)     (0.0120)         
AGE2               -0.0020    -0.0020      -0.0020          
                   (0.0000)   (0.0000)     (0.0002)         
Intercept          -2.7351    -2.7351      -2.7351          
                   (0.0131)   (0.0109)     (0.1594)         
BLACK              -0.1